# 0) Mental model

* **Type**: dynamic array (contiguous memory), **mutable**, **ordered**, **0-indexed**.
* Negative index: `a[-1]` → last element.

# 1) Create

```python
a = [1, 2, 3]                 # literal
b = list((4, 5))              # from tuple/iterable
c = list(range(5))            # [0,1,2,3,4]
d = [0] * 5                   # repetition
e = [[0]*3 for _ in range(4)] # 4x3 grid (safe)
```

⚠️ `[[0]*3]*4` makes 4 references to the **same** inner list (aliasing bug).

# 2) Read / Access

```python
a[0], a[-1]                   # first, last
a[1:4]                        # slice (new list)
a[:], a[::-1], a[::2]         # copy, reverse view, step
```

Slicing never errors on bounds; indexing does.

# 3) Update (mutate)

```python
a[1] = 99                     # single index assign
a[1:3] = [7, 8, 9]            # replace (length may differ)
a[1:1] = [100, 101]           # insert at index 1
a[1:4] = []                   # delete slice
a[::2] = [9, 9, 9]            # extended-slice assign: lengths must match
```

# 4) Add elements

```python
a.append(x)                   # add one item to end
a.extend(iterable)            # add many (flattens one level)
a += iterable                 # same as extend
a.insert(i, x)                # O(n) shift right
a = [*a, *more]               # unpack into new list
```

`append([1,2])` keeps the list as a **single** element; `extend([1,2])` adds two elements.

# 5) Remove elements

```python
x = a.pop()                   # remove & return last
x = a.pop(i)                  # remove & return at i
a.remove(value)               # first occurrence of value (ValueError if absent)
del a[i]                      # delete by index
del a[i:j:k]                  # delete slice/stride
a.clear()                     # empty list
```

# 6) Search / Check

```python
i = a.index(x)                # first index of x
i = a.index(x, start, end)    # bounded search
cnt = a.count(x)              # frequency
found = (x in a)              # membership (linear scan)
```

# 7) Sort / Reverse

```python
a.sort()                      # in-place, stable
a.sort(reverse=True)          # descending
a.sort(key=str.lower)         # custom key
b = sorted(a)                 # new list (original untouched)
a.reverse()                   # in-place reverse
rev_iter = reversed(a)        # iterator (no mutation)
```

**Stable sort**: equal keys keep relative order—useful for multi-key sorting via chained sorts or tuples: `a.sort(key=lambda t: (t[0], -t[1]))`.

# 8) Copying (important)

```python
b = a                  # alias (same object)
b = a[:]               # shallow copy
b = list(a)            # shallow copy
b = a.copy()           # shallow copy (same as above)
import copy; b = copy.deepcopy(a)   # deep copy (nested structures copied)
```

Shallow copies duplicate the outer list only.

# 9) Iterate & transform

```python
for x in a: ...
for i, x in enumerate(a, start=0): ...
for x in reversed(a): ...

# Transformations
b = [f(x) for x in a]                   # comprehension
b = [x for x in a if pred(x)]           # filter + map
b = list(map(f, a))
b = list(filter(pred, a))
```

# 10) Aggregate & utilities

```python
n = len(a); s = sum(a)                  # size, sum
mn, mx = min(a), max(a)
ok = any(a), all(a)                     # truthiness over items
t = tuple(a)                            # to tuple (immutable)
a = list("abc")                         # from iterable of chars
" ".join(list_of_strings)               # join (all must be str)
```

# 11) Combine, zip, unzip

```python
c = a + b                               # new list
c = [*a, *b]                            # new list via unpack
pairs = list(zip(a, b))                 # zip
xs, ys = map(list, zip(*pairs))         # unzip
```

# 12) Queue / Stack / Heap patterns

```python
# Stack
stack = []
stack.append(x); x = stack.pop()

# Queue (avoid list.pop(0) — O(n)); use deque
from collections import deque
q = deque([1,2]); q.append(3); x = q.popleft()

# Priority queue
import heapq
h = []; heapq.heappush(h, (priority, item)); priority, item = heapq.heappop(h)
```

# 13) Common recipes (loop or pythonic)

**Deduplicate (preserve order)**

```python
seen=set(); out=[]
for x in a:
    if x not in seen:
        seen.add(x); out.append(x)
```

**Partition by predicate (stable)**

```python
neg = [x for x in a if pred(x)]
pos = [x for x in a if not pred(x)]
out = neg + pos
```

**Rotate right by k (O(1) extra using reverse trick)**

```python
k %= len(a)
a.reverse(); a[:k] = reversed(a[:k]); a[k:] = reversed(a[k:])
```

**Merge two sorted lists (no sort at end)**

```python
i=j=0; out=[]
while i<len(a) and j<len(b):
    if a[i] <= b[j]: out.append(a[i]); i+=1
    else: out.append(b[j]); j+=1
out.extend(a[i:]); out.extend(b[j:])
```

**Flatten one level**

```python
flat = [y for x in lol for y in x]
```

**Chunk into size n**

```python
chunks = [a[i:i+n] for i in range(0, len(a), n)]
```

**In-place remove all zeros (no extra list)**

```python
write = 0
for read in range(len(a)):
    if a[read] != 0:
        a[write] = a[read]; write += 1
del a[write:]
```

# 14) Performance (amortized/typical)

* `append`, `pop()` at end: **O(1)** amortized
* Index access `a[i]`: **O(1)**
* `insert(i, x)`, `pop(0)`, `del a[i]`: **O(n)** (shift elements)
* `x in a`, `index`, `remove`: **O(n)**
* `sort`: **O(n log n)** (Timsort, stable)
* Slice copy `a[:]`: **O(n)**

# 15) Pitfalls & best practices

* Don’t mutate while iterating over the same list; iterate over a **copy** or write with two-pointer pattern.
* Beware `[[0]*m]*n` aliasing; use a list-comp to build nested lists.
* `list` ≠ NumPy array: lists don’t do vectorized math (`[1,2]+[3,4]` concatenates).
* For frequent front ops, use `deque`, not a list.

# 16) Full methods reference (quick glance)

* **Mutation**: `append`, `extend`, `insert`, `remove`, `pop`, `clear`, `reverse`, `sort`
* **Query**: `index`, `count`
* **Copy**: `copy` (shallow)
* **Operators**: `+`, `*`, `in`, slicing `[:]`, `del`, unpacking `*iterable`



In [4]:
a = [1,2,3]

In [8]:
b = list((4,5))

In [10]:
c = list(range(4))

In [14]:
d = [2]*3

In [16]:
d

[2, 2, 2]

In [28]:
e = [[0]*3 for i in range(4)]

In [30]:
e

[[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]

In [36]:
a[0]; a[-1]

3

In [38]:
a[1:4]

[2, 3]

In [42]:
a[1] =99

In [44]:
a

[1, 99, 3]

In [46]:
a[1:3] = [7,8,9]

In [48]:
a

[1, 7, 8, 9]

In [52]:
a[1:1] = [100,101]

In [54]:
a

[1, 100, 101, 7, 8, 9]

Awesome—let’s kick off a focused “List Gym.”
Here’s a ladder of **practice questions** (from basics → interview-grade). Do them in order; most have constraints to force clean logic.

# Python Lists — Practice Set 1

## A) Warm-up & Indexing

1. Create a list `a` from `1..10`. Print: first, last, middle, last 3, every 2nd.
2. Given `a=[3,1,4,1,5,9]`, show three ways to get a **reversed** view/result (note which mutate).
3. Concatenate `x=[1,2]` and `y=[3,4]` two ways. Which creates a new list?
4. Membership: check if `9` in `a`. If present, print its **first** and **last** index (without `.index()` on last).
5. Slice assign: turn `a=[0,0,0,0]` into `[1,2,3,4]` using **one** slice assignment.

## B) Mutability & Core Methods

6. Show the difference between `append` vs `extend` using `lst=[1,2]` and `to_add=[3,4]`.
7. Use `insert` to put `0` at the start of `[1,2,3]`.
8. Remove only the **first** occurrence of `2` from `[1,2,3,2,4]` (without list comp).
9. Pop and store the last element of `[10,20,30]` into `last`, leaving `[10,20]`.
10. In-place reverse vs `reversed(...)` iterator: demonstrate both on `[1,2,3]`.
11. Sort `people=["bob","Alice","charlie"]` case-insensitively but **do not** mutate the original.
12. Sort `words=["a","bbb","cc"]` by descending length; if equal length, ascending lexicographic.
13. Stable sort demo: sort `[(2,"a"),(1,"b"),(2,"c")]` by first key, preserving relative order of equal keys—verify stability.
14. Clear a list in three ways; which one rebinds vs mutates?
15. Shallow copy: make three distinct shallow copies of `a` without using `copy.deepcopy`.

## C) Copying Traps (Shallow vs Deep)

16. Why is `rows = [[0]*3]*4` dangerous? Show the bug by setting `rows[0][0]=1`. Fix it two ways.
17. Given `m = [[1,2],[3,4]]`, make a **deep** copy and change only the inner list of the copy. Prove the original is unchanged.

## D) Frequency, Filtering, and In-place Editing

18. Count occurrences of each value in `a=[1,2,2,3,3,3]` **without** `collections.Counter`. Output dict `{1:1,2:2,3:3}`.
19. Remove **all** `0`s from `a=[0,1,0,2,0,3]` in-place (no extra list, no list comp).
20. Move all zeros to the **end** in-place, preserving the order of non-zeros.
21. Deduplicate while preserving order: `[3,1,3,2,1] → [3,1,2]` (no set in the final result ordering step that loses order).
22. Find the **second largest** distinct element in `[2,7,7,3,5]` (handle fewer than 2 distinct values).
23. Merge two **sorted** lists into one sorted list without `+` and without `.sort()` at the end.
24. Rotate a list right by `k` in-place: `[1,2,3,4,5], k=2 → [4,5,1,2,3]` (O(1) extra space).

## E) Sliding Window on Lists

25. Max sum of any subarray of size `k` (positive/negative allowed). Return the sum and start index.
26. Smallest subarray length with sum ≥ `S` (positive integers). Return length (or 0 if none).
27. Longest subarray with **all unique** elements. Return start, end, length.
28. Count subarrays of size `k` whose average ≥ `T` (single pass if possible).

## F) Search & Two-Pointers (List-centric)

29. Two-Sum (return indices) for `a` **unsorted**—two solutions:

* a) O(n²) with loops only
* b) O(n) with a dict (no sorting).

30. Given a **sorted** list, remove duplicates in-place and return the new logical length.
31. Check if `a` is a palindrome using two pointers (in-place, O(1) extra).
32. Partition: rearrange `a` so that all `< pivot` come before `>= pivot` (single pass).

## G) Nested Lists / “Matrix as List-of-Lists”

33. Transpose a matrix (no NumPy):

```
[[1,2,3],
 [4,5,6]] → [[1,4],[2,5],[3,6]]
```

34. Rotate matrix 90° clockwise in-place (square matrix).
35. Spiral order of a matrix—return the traversal as a flat list.
36. Set Matrix Zeroes: if any cell is 0, set its row & column to 0 (O(1) extra if you can).
37. Pascal’s Triangle (first `n` rows) using only lists.

## H) “Why” / Theory Checks (Short Answers)

38. List vs tuple vs NumPy array—mutability, memory layout, arithmetic semantics, speed.
39. What is copied by `b = a`, `b = a[:]`, `b = list(a)`, and `copy.deepcopy(a)`?
40. Big-O (average): `append`, `pop()`, `insert(0,x)`, `pop(0)`, `x in a`, `sort`, `slice copy a[:]`.
41. Why is `list` a poor choice for queue operations at the front? What should you use instead?
42. Explain Timsort and why Python’s sort is **stable** (give a scenario where stability matters).

---

If you want, say **“Give solutions for A–D only”** (or any block), and I’ll walk you through clean loop-based answers first, then show tasteful Pythonic alternatives.


In [58]:
a = [3,1,4,1,5,9]

rev_slice = a[::-1]


In [60]:
a.reverse()

In [62]:
a

[9, 5, 1, 4, 1, 3]

In [66]:
s = reversed(a)

In [70]:
list(s)

[3, 1, 4, 1, 5, 9]

In [72]:
x = [2,3]
y = [3,4]

z1 = x + y

In [74]:
z1

[2, 3, 3, 4]

In [76]:
z2 = [*x, *y]

In [78]:
z2

[2, 3, 3, 4]

In [80]:
x.extend(y)

In [82]:
x

[2, 3, 3, 4]

In [84]:
x.extend(y)

In [86]:
x

[2, 3, 3, 4, 3, 4]

In [90]:
a = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 9]

if 9 in a:
    first = a.index(9)

    last = None
    for i in range(len(a)-1, -1, -1):
        if a[i] == 9:
            last = i
            break
    
    print(f"first index: {first}, last index: {last}")
else:
    print("9 not found")


first index: 5, last index: 11


In [103]:
if 7 in a:
    f = a.index(7)
    l = None
    for i in range(len(a)-1,-1,-1):
        if a[i] == 7:
            last = i
            break
    print(f"{f}{l}")
else:
    print('not')

not


In [105]:
a = [0,0,0,0]
a[:] = [1,2,3,4]
print(a)

[1, 2, 3, 4]


In [107]:
# . Show the difference between `append` vs `extend` using `lst=[1,2]` and `to_add=[3,4]`.



In [109]:
import torch, torch.nn as nn, torch.optim as optim

class TinyNet(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden), nn.ReLU(),
            nn.Linear(d_hidden, d_out)
        )
    def forward(self, x): return self.net(x)

model = TinyNet(d_in=100, d_hidden=64, d_out=10)
opt = optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

# x: [batch, 100], y: [batch] class ids
for x, y in dataloader:
    opt.zero_grad()
    loss = loss_fn(model(x), y)
    loss.backward()
    opt.step()


ModuleNotFoundError: No module named 'torch'

In [111]:
# Function and derivative
def f(x):
    return (x - 3)**2

def f_prime(x):   # derivative = 2(x-3)
    return 2*(x - 3)

# Initialize
x = 0        # starting point
lr = 0.1     # learning rate
epochs = 20  # number of steps

for i in range(epochs):
    grad = f_prime(x)        # compute slope
    x = x - lr * grad        # update rule
    print(f"Step {i+1}: x={x:.4f}, f(x)={f(x):.4f}")


Step 1: x=0.6000, f(x)=5.7600
Step 2: x=1.0800, f(x)=3.6864
Step 3: x=1.4640, f(x)=2.3593
Step 4: x=1.7712, f(x)=1.5099
Step 5: x=2.0170, f(x)=0.9664
Step 6: x=2.2136, f(x)=0.6185
Step 7: x=2.3709, f(x)=0.3958
Step 8: x=2.4967, f(x)=0.2533
Step 9: x=2.5973, f(x)=0.1621
Step 10: x=2.6779, f(x)=0.1038
Step 11: x=2.7423, f(x)=0.0664
Step 12: x=2.7938, f(x)=0.0425
Step 13: x=2.8351, f(x)=0.0272
Step 14: x=2.8681, f(x)=0.0174
Step 15: x=2.8944, f(x)=0.0111
Step 16: x=2.9156, f(x)=0.0071
Step 17: x=2.9324, f(x)=0.0046
Step 18: x=2.9460, f(x)=0.0029
Step 19: x=2.9568, f(x)=0.0019
Step 20: x=2.9654, f(x)=0.0012


In [113]:
def f(x):
    return (x-3)**2
def f_prime(x):
    return 2*(x-3)

x = 0
lr = 0.1
itr = 20

for i in range(itr):
    grad = f_prime(x)
    x = x - lr * grad
    print(f"step {i+1}: x={x:.4f}, f(x)={f(x):.4f}")



step 1: x=0.6000, f(x)=5.7600
step 2: x=1.0800, f(x)=3.6864
step 3: x=1.4640, f(x)=2.3593
step 4: x=1.7712, f(x)=1.5099
step 5: x=2.0170, f(x)=0.9664
step 6: x=2.2136, f(x)=0.6185
step 7: x=2.3709, f(x)=0.3958
step 8: x=2.4967, f(x)=0.2533
step 9: x=2.5973, f(x)=0.1621
step 10: x=2.6779, f(x)=0.1038
step 11: x=2.7423, f(x)=0.0664
step 12: x=2.7938, f(x)=0.0425
step 13: x=2.8351, f(x)=0.0272
step 14: x=2.8681, f(x)=0.0174
step 15: x=2.8944, f(x)=0.0111
step 16: x=2.9156, f(x)=0.0071
step 17: x=2.9324, f(x)=0.0046
step 18: x=2.9460, f(x)=0.0029
step 19: x=2.9568, f(x)=0.0019
step 20: x=2.9654, f(x)=0.0012


In [117]:
a = [1,2]
a.insert(0,1)

In [121]:
a.extend([535])

In [129]:
b =  [1,2,35,6]


In [131]:
a.extend(b)

In [133]:
a

[1, 1, 2, 535, 4, 5, 1, 2, 35, 6]

In [139]:
list1 = [1,2,3,2,4]
list1.remove(2)

In [143]:
list1

[1, 3, 2, 4]

In [147]:
a = [23,23,32,32]

b = a.pop()

In [149]:
b

32